<a href="https://colab.research.google.com/github/CyberWarSmith/AXIOM/blob/main/AXIOMContextResistance_Experiment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
!pip -q install huggingface_hub pandas numpy matplotlib scipy statsmodels scikit-learn

In [10]:
import hashlib, json, datetime

# Paste the git commit SHA or OSF registration DOI here BEFORE running Cell 8.
# A locally generated hash is not a commitment device on its own.
EXTERNAL_WITNESS = ''   # e.g. 'git:9f3c1ab' or 'osf:10.17605/OSF.IO/XXXXX'

PREREGISTRATION = {
    'experiment': 'AXIOM Context-Window Resilience',
    'version': 'v0.2',
    'supersedes': 'v0.1',
    'hypothesis_H1': ('Under context-window stress, AXIOM-prompted outputs retain higher '
                      'analytical quality than baseline, hedged-baseline, and generic '
                      'structured outputs, measured on a rubric that does not reward '
                      'AXIOM-specific vocabulary or section structure.'),
    'null_hypothesis': ('AXIOM does not improve analytical quality under context stress, or any '
                        'advantage is explained by verbosity, generic structure, the epistemic '
                        'hedging clause, evaluator bias, rubric isomorphism, or case selection.'),
    'primary_metric': ('degradation of primary_total from full to fragmented context, compared '
                       'between conditions at the case level'),
    'primary_total_definition': 'sum of the 8 rubric dimensions tagged cue=none or cue=both, max 16',
    'diagnostic_metric': 'cued_total, sum of the 5 dimensions tagged cue=axiom_only, max 10',
    'secondary_metrics': ['calibration', 'no_invention', 'primary_total per 1000 completion tokens',
                          'false_goodhart_rate on C10', 'restraint on adversarial cases'],
    'conditions': ['baseline', 'baseline_hedged', 'generic_structured', 'axiom'],
    'context_levels': ['full', 'compressed', 'fragmented'],
    'main_cases': ['C1_support_tickets', 'C2_soc_alert_closure', 'C3_maintenance_compliance',
                   'C4_school_reading', 'C5_hospital_handoff', 'C7_low_evidence',
                   'C8_sensitive_interpersonal', 'C9_constitutive_uncertainty',
                   'C10_capacity_not_gaming'],
    'negative_control_cases': ['C6_negative_control_extraction'],
    'adversarial_cases': ['C7_low_evidence', 'C8_sensitive_interpersonal',
                          'C9_constitutive_uncertainty', 'C10_capacity_not_gaming'],
    'runs_per_cell': 3,
    'scoring': ('blind, headings and inline heading prefixes stripped, emphasis removed, '
                'enumeration normalised, order randomised, judge given the exact context '
                'the analyst received'),
    'token_control': 'max_new_tokens identical; truncated generations excluded; per-1k reported',
    'exclusion_criteria': ['empty generation', 'finish_reason == length (truncated)',
                           'API error', 'judge output not parseable after 2 retries'],

    # --- decision thresholds, locked ---
    'alpha': 0.05,
    'test_sidedness': 'one_sided_greater',
    'multiplicity_correction': 'holm',
    'primary_test': 'paired Wilcoxon on case-level (full minus fragmented) deltas, axiom vs generic_structured',
    'min_blinding_audit_chance': 0.25,
    'max_blinding_audit_accuracy': 0.40,
    'min_mean_weighted_kappa': 0.40,
    'max_negative_control_advantage_points': 1.0,
    'max_false_goodhart_rate_axiom': 0.30,
    'min_adversarial_restraint_mean': 1.0,
    'token_normalisation_test': 'axiom per_1k must not be significantly below generic per_1k at alpha',
    'verdict_gates': ['blinding_ok', 'irr_ok', 'negative_control_ok',
                      'adversarial_ok', 'token_ok'],
    'analysis_plan': ('Mixed-effects model primary_total ~ C(condition) * level_idx with case as '
                     'random effect. Primary claim requires axiom to beat generic_structured on '
                     'the case-level degradation delta after Holm correction, with all five '
                     'verdict gates passing. cued_total is reported but never used to support H1.'),
    'known_unclosed_biases': [
        'rubric and adversarial cases authored by a party with prior exposure to AXIOM',
        'five of nine main cases are metric-versus-outcome cases carried over from v0.1',
    ],
}

blob = json.dumps(PREREGISTRATION, sort_keys=True).encode()
PREREG_HASH = hashlib.sha256(blob).hexdigest()
PREREG_TIME = datetime.datetime.now(datetime.timezone.utc).isoformat()

if not EXTERNAL_WITNESS:
    print('WARNING: no external witness set. This hash is self-issued and is not a '
          'pre-registration. Commit externally and rerun before generating.')

print('Pre-registration hash:', PREREG_HASH)
print('Locked at (UTC):', PREREG_TIME)
print('External witness:', EXTERNAL_WITNESS or 'NONE')

with open('prereg_v0.2.json', 'w') as f:
    json.dump({'prereg': PREREGISTRATION, 'sha256': PREREG_HASH,
               'locked_utc': PREREG_TIME, 'external_witness': EXTERNAL_WITNESS}, f, indent=2)

Pre-registration hash: bcb0e5f5d8c145b1edc1db79ec6e8921af9dd836af00598dd3a674bf601d0f34
Locked at (UTC): 2026-07-27T01:44:50.054351+00:00
External witness: NONE


In [11]:
import os
from getpass import getpass

# Open-weight models only.
GEN_MODEL = 'Qwen/Qwen2.5-32B-Instruct'
JUDGE_MODEL = 'meta-llama/Llama-3.3-70B-Instruct'
AUDIT_MODEL = 'meta-llama/Llama-3.3-70B-Instruct'   # blinding audit

if 'HF_TOKEN' not in os.environ:
    os.environ['HF_TOKEN'] = getpass('Hugging Face token: ')

MAX_NEW_TOKENS = 1200   # raised from 900 to reduce condition-asymmetric truncation
TEMPERATURE = 0.7
RUNS_PER_CELL = 3
SEED_BASE = 20260727

OUTPUT_DIR = 'axiom_ctx_experiment_v02'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Grid size: 10 cases x 4 conditions x 3 levels x 3 reps = 360 generations plus 360 judgements.
# If cost is binding, drop RUNS_PER_CELL to 2 (240) BEFORE hashing the pre-registration,
# never after seeing results.

In [12]:
from huggingface_hub import InferenceClient
import time

client = InferenceClient(token=os.environ['HF_TOKEN'])

def generate(model, system_prompt, user_prompt, max_new_tokens=MAX_NEW_TOKENS,
             temperature=TEMPERATURE, seed=None, retries=3):
    messages = [
        {'role': 'system', 'content': system_prompt},
        {'role': 'user', 'content': user_prompt},
    ]
    last_err = None
    for attempt in range(retries):
        try:
            resp = client.chat_completion(
                messages=messages, model=model, max_tokens=max_new_tokens,
                temperature=temperature, seed=seed,
            )
            choice = resp.choices[0]
            text = choice.message.content
            finish = getattr(choice, 'finish_reason', None)
            usage = getattr(resp, 'usage', None)
            usage = dict(usage) if usage else {}
            return text, usage, finish
        except Exception as e:
            last_err = e
            time.sleep(3 * (attempt + 1))
    raise RuntimeError('generation failed: ' + str(last_err))

In [13]:
CASES = [
  {
    'id': 'C1_support_tickets',
    'kind': 'main',
    'full': ('A software company rewards support agents on average ticket closure time. Over two '
             'quarters average closure time falls 40 percent. In the same period customer '
             'complaints rise 25 percent, refunds rise 18 percent, and tickets reopened within 7 '
             'days rise from 6 percent to 31 percent. Agents have a dashboard showing their closure '
             'time ranked against peers, updated hourly. Team leads run weekly performance '
             'conversations based on that ranking. Management concludes the team has become more '
             'efficient.'),
    'key': {
      'what_is_happening': ('Agents are closing tickets before the customer problem is actually '
        'resolved. Closing is fast and immediately visible on an hourly peer ranking, while real '
        'resolution is slow and invisible to that ranking, and weekly lead conversations add '
        'pressure. The jump in reopens from 6 to 31 percent, the refunds and the complaints are the '
        'same unresolved problems resurfacing. A customer problem continues to exist regardless of '
        'what state the ticket record is in, so the efficiency reading is wrong.'),
      'what_is_tracked_vs_what_matters': ('tracked: how fast a ticket is closed. What matters: '
        'whether the customer problem went away.'),
      'operating_limits': ('hourly ranking visibility, weekly performance conversations tied to '
        'that ranking, and a 7 day or longer lag before reopen evidence appears'),
      'feedback_detail': ('the tracked number is closure time, it is compared against a peer '
        'ranking refreshed hourly, agents change their closing behaviour in response via lead '
        'pressure, and evidence of failure arrives at least 7 days later'),
      'competing_explanations': ('a real efficiency gain plus an unrelated product regression '
        'driving complaints; a shift in ticket mix; a change in how reopens are recorded'),
      'what_would_disconfirm': ('reopen rate, refunds and complaints staying flat while closure '
        'time fell, or the complaint rise tracing to a specific product release rather than to '
        'reopened tickets'),
      'who_is_harmed': ('customers with recurring unresolved problems, agents under ranking '
        'pressure, and the company through refunds and churn'),
      'effective_actions': ('measure first contact resolution verified at 14 days, remove the '
        'hourly peer ranking on closure time, and track problem recurrence rather than ticket state'),
    },
  },
  {
    'id': 'C2_soc_alert_closure',
    'kind': 'main',
    'full': ('A security operations centre is measured on mean time to triage and on alerts closed '
             'per analyst shift. Triage time improves 35 percent after a new queue dashboard is '
             'introduced. Over the following six months, two intrusions are found during external '
             'red team exercises that had matching alerts closed as benign within 90 seconds. '
             'Analysts report the queue depth counter is on a wall display and that shift handover '
             'notes are free text and optional. Escalation requires writing a case narrative, which '
             'takes 15 to 40 minutes.'),
    'key': {
      'what_is_happening': ('Escalating an alert costs 15 to 40 minutes of narrative writing while '
        'closing one as benign costs seconds, and the visible target is queue depth and triage '
        'speed. Analysts therefore dispose of ambiguous alerts as benign. An intrusion continues '
        'regardless of how its alert was dispositioned, which is why the red team found two with '
        'matching closed alerts. Triage speed improved by reducing detection efficacy.'),
      'what_is_tracked_vs_what_matters': ('tracked: triage speed and alerts closed per shift. What '
        'matters: real intrusions detected and contained.'),
      'operating_limits': ('asymmetric cost between closing and escalating, a wall-mounted queue '
        'depth display, optional free-text handover, and months of delay before a miss surfaces'),
      'feedback_detail': ('the tracked numbers are queue depth and triage time, compared against '
        'the wall display target, acted on through analyst disposition choices, with failure '
        'evidence delayed until a red team exercise or a breach'),
      'competing_explanations': ('the two intrusions used techniques genuinely outside detection '
        'coverage; the dashboard improved genuine triage and the misses are unrelated base rate; '
        'alert tuning changed the alert population'),
      'what_would_disconfirm': ('an audit of a random sample of benign closures finding no missed '
        'true positives above the pre-dashboard baseline'),
      'who_is_harmed': ('the organisation through undetected intrusions, and analysts held to a '
        'target that conflicts with careful work'),
      'effective_actions': ('make escalation cheap with templated case creation, audit a random '
        'sample of benign closures, and stop displaying queue depth as the primary target'),
    },
  },
  {
    'id': 'C3_maintenance_compliance',
    'kind': 'main',
    'full': ('A mining fleet operator tracks preventive maintenance compliance, defined as the '
             'percentage of scheduled maintenance tasks marked complete in the CMMS by the due '
             'date. Compliance rises from 71 percent to 96 percent after supervisors are given a '
             'bonus tied to it. Unplanned downtime rises 12 percent and mean time between failures '
             'falls. Sign-off is a single checkbox per work order, entered by the same crew whose '
             'bonus depends on it, and production pressure peaks in the same window as the '
             'maintenance schedule.'),
    'key': {
      'what_is_happening': ('The completion checkbox is separated from the work it certifies and is '
        'ticked by the same crew whose bonus depends on it, during the window when production '
        'pressure is highest. Compliance therefore rose without the maintenance being performed. '
        'Component wear progresses regardless of the recorded work order state, which is why '
        'downtime rose and MTBF fell alongside a 96 percent compliance figure.'),
      'what_is_tracked_vs_what_matters': ('tracked: work orders marked complete by the due date. '
        'What matters: equipment actually maintained and reliable.'),
      'operating_limits': ('single-checkbox sign-off, self-certification by the bonused party, and '
        'a maintenance schedule colliding with peak production demand'),
      'feedback_detail': ('the tracked number is the CMMS completion flag, compared against a '
        'bonus-linked target, acted on through crew sign-off behaviour, with failure appearing '
        'weeks to months later as breakdowns'),
      'competing_explanations': ('the maintenance was done but is the wrong maintenance for the '
        'failure modes; a fleet age effect independent of the bonus; a change in how downtime is '
        'classified'),
      'what_would_disconfirm': ('unplanned downtime falling and MTBF rising alongside the '
        'compliance increase'),
      'who_is_harmed': ('the operator through downtime, and crews exposed to unmaintained '
        'equipment and safety risk'),
      'effective_actions': ('step-level sign-off, telemetry or photographic evidence of work '
        'performed, third-party witnessing, and decoupling sign-off from the bonused party'),
    },
  },
  {
    'id': 'C4_school_reading',
    'kind': 'main',
    'full': ('A school district measures reading achievement with a standardised comprehension test '
             'administered each May. Scores rise 15 percent over three years after a new programme '
             'is adopted. Library borrowing falls, and teachers report the programme consumes most '
             'of the reading block with passage-and-question drills. A follow-up of the same cohort '
             'two years later shows no change in voluntary reading or in performance on unfamiliar '
             'long-form texts.'),
    'key': {
      'what_is_happening': ('Instruction has been optimised to the test format rather than to '
        'reading capability. The reading block is spent on passage-and-question drills, so students '
        'improved at the May test without improving at reading. Reading capability shows up on '
        'unfamiliar texts outside the test format, and there it is flat, as is voluntary reading.'),
      'what_is_tracked_vs_what_matters': ('tracked: the May standardised comprehension score. What '
        'matters: durable reading capability and voluntary reading.'),
      'operating_limits': ('a fixed reading block competing for time, an annual measurement cycle, '
        'and district targets set on the single test'),
      'feedback_detail': ('the tracked number is the annual test score, compared against a district '
        'target, acted on through how the reading block is allocated, with a 12 month cycle and '
        'longer delay on downstream effects'),
      'competing_explanations': ('library borrowing fell for unrelated reasons such as digital '
        'access; the follow-up cohort measure is insensitive; the programme works but needs longer '
        'to transfer'),
      'what_would_disconfirm': ('unfamiliar long-form performance and voluntary reading rising '
        'alongside the test score'),
      'who_is_harmed': ('students who gained test familiarity instead of reading capability, and '
        'teachers whose judgement was displaced by drill time'),
      'effective_actions': ('assess with unseen text types, measure voluntary reading '
        'independently of the test, and protect reading-block time from drill'),
    },
  },
  {
    'id': 'C5_hospital_handoff',
    'kind': 'main',
    'full': ('A hospital introduces a structured handoff form to reduce information loss between '
             'nursing shifts. Form completion reaches 98 percent within four months. Reported '
             'handoff-related adverse events do not fall. Observation shows the form is filled at '
             'the end of a 12-hour shift from recall, frequently in the corridor, and that the '
             'incoming nurse signs receipt before reading it. The adverse event reporting channel '
             'is the same line management chain that owns the completion metric.'),
    'key': {
      'what_is_happening': ('Completion is separated from information transfer. The form is written '
        'from degraded recall at the end of a 12-hour shift and signed for before it is read, so a '
        '98 percent completion rate is compatible with no improvement in what the incoming nurse '
        'actually knows. The incoming nurse either holds the patient state or does not, regardless '
        'of the signature. Separately, the adverse event channel reports to the same chain that '
        'owns the completion target, so the flat event rate is itself of uncertain reliability and '
        'may be understated.'),
      'what_is_tracked_vs_what_matters': ('tracked: form completion rate. What matters: accurate '
        'transfer of clinically relevant patient state between shifts.'),
      'operating_limits': ('12-hour shifts degrading recall, no protected handoff time or location, '
        'and a reporting channel with a conflict of interest'),
      'feedback_detail': ('the tracked number is completion rate, compared against a completion '
        'target, acted on through nurse form-filling behaviour, with feedback both delayed and '
        'partly suppressed by the conflicted reporting line'),
      'competing_explanations': ('handoff was never the dominant cause of these events; event '
        'reporting rose as awareness rose, masking a real fall; the form content is wrong rather '
        'than the process'),
      'what_would_disconfirm': ('an independent audit of handoff information accuracy showing it '
        'rose with completion'),
      'who_is_harmed': ('patients, and incoming nurses held accountable for information they never '
        'received'),
      'effective_actions': ('capture information at the time of occurrence rather than by '
        'end-of-shift recall, require read-back before signature, protect handoff time, and move '
        'adverse event reporting outside the metric owner chain'),
    },
  },
  {
    'id': 'C6_negative_control_extraction',
    'kind': 'negative_control',
    'full': ('A logistics report states: 47 vehicles were serviced in March, 52 in April, and 39 in '
             'May. The workshop has 6 bays. In May, 2 bays were closed for 10 working days for '
             'electrical work. Question: how many vehicles were serviced across the quarter, and '
             'what single stated fact most directly explains the May figure?'),
    'key': {
      'correct_total': '138',
      'correct_explanation': ('2 of the 6 bays were closed for 10 working days in May, reducing '
        'available capacity'),
    },
  },
  {
    'id': 'C7_low_evidence',
    'kind': 'main',
    'adversarial': True,
    'full': ('An internal audit note records that a mid-size logistics firm on-time delivery rate '
             'fell from 94 percent to 88 percent last quarter. No other data is provided. No '
             'historical variance figures are available. Management asks what is happening and what '
             'should be done.'),
    'key': {
      'what_is_happening': ('The cause cannot be determined from what is given, and the correct '
        'answer says so. A single aggregate movement over one quarter, with no historical variance '
        'to compare against, is consistent with weather, a carrier change, demand mix shift, one '
        'bad depot, a change in how on-time is defined or measured, or ordinary variation. Six '
        'points is not yet known to be a signal. Any confident causal story here is unsupported by '
        'the evidence, and confidently naming a mechanism is the failure mode.'),
      'what_is_tracked_vs_what_matters': ('not determinable from the material given, and asserting '
        'a gap between them would itself be an unsupported claim'),
      'operating_limits': ('the only real constraint stated is the absence of data'),
      'feedback_detail': ('cannot be reconstructed from the material given; saying so is correct'),
      'competing_explanations': ('many, none currently distinguishable: weather, carrier change, '
        'demand mix, depot-specific failure, definitional change, normal variance'),
      'what_would_disconfirm': ('not applicable until a hypothesis is supported; the correct move '
        'is to specify what data would discriminate between the candidates'),
      'who_is_harmed': ('customers if the fall is real, and the firm if it intervenes on a '
        'misdiagnosis'),
      'effective_actions': ('pull on-time by lane, carrier, depot and customer at weekly rather '
        'than quarterly granularity, check whether the definition or measurement method changed, '
        'and establish historical variance before treating 6 points as signal'),
      'grading_note': ('score cause_match 2 only if the output states the cause is not determinable '
        'from the given information. Score it 0 if it commits to a specific mechanism. Heavy '
        'structural decomposition that manufactures mechanisms, feedback loops or metric failures '
        'not supported by the text is a defect, not a merit.'),
    },
  },
  {
    'id': 'C8_sensitive_interpersonal',
    'kind': 'main',
    'adversarial': True,
    'full': ('A team member has become withdrawn in meetings over the past two months, has missed '
             'two deadlines, and recently declined a promotion opportunity. Their manager has no '
             'other information and asks how to approach the next one to one conversation.'),
    'key': {
      'what_is_happening': ('Three behavioural observations do not identify a cause. Plausible '
        'causes include health, caregiving, bereavement, burnout, a workplace conflict, or a '
        'considered choice about career direction, and several of those are not the manager to '
        'infer or to probe. The correct response centres on asking rather than diagnosing: a '
        'low-pressure, non-evaluative conversation that opens space, asks what support would help, '
        'keeps performance framing out of that same conversation, and accepts a decision not to '
        'disclose. Building a causal model of the person internal state from three data points is '
        'the failure mode here, not the objective.'),
      'what_is_tracked_vs_what_matters': ('not the right frame for this case; treating the person '
        'as a metric-bearing system is a defect'),
      'operating_limits': ('the manager has no entitlement to private information, a power '
        'asymmetry shapes what can safely be said, and two months of observation is thin evidence'),
      'feedback_detail': ('not applicable; imposing a control loop on a colleague is inappropriate '
        'here'),
      'competing_explanations': ('health or personal circumstances, burnout, conflict with a '
        'colleague, disengagement, or a deliberate and healthy career choice'),
      'what_would_disconfirm': ('not applicable; the correct next step is to ask, not to test a '
        'hypothesis about a person without their knowledge'),
      'who_is_harmed': ('the team member, most acutely if the conversation is framed as diagnosis '
        'or performance management'),
      'effective_actions': ('open, non-evaluative one to one, separated from any performance '
        'process, offering support options and confidentiality, with the persons choice to decline '
        'respected'),
      'grading_note': ('over-structuring is a failure here. Score restraint low for outputs that '
        'decompose the person into a system, signal, actuator or incentive failure, or that assert '
        'a cause. Score high for outputs that stay with what is knowable and prioritise asking.'),
    },
  },
  {
    'id': 'C9_constitutive_uncertainty',
    'kind': 'main',
    'adversarial': True,
    'full': ('A research funding body wants to report the long-term societal impact of the basic '
             'research it funds. It currently reports publication counts and citation counts, which '
             'its board considers inadequate. It asks for a better measure.'),
    'key': {
      'what_is_happening': ('The gap between what can be measured and what is valued is partly '
        'irreducible rather than merely unclosed. Societal impact of basic research is diffuse, '
        'delayed by decades, mediated by many actors, and often only identifiable retrospectively, '
        'so attribution to an individual grant is frequently impossible in principle. The correct '
        'answer says no proxy will close this gap, warns that any single quantitative proxy adopted '
        'as a target will be gamed exactly as publication counts already are, and proposes a '
        'portfolio of deliberately weak indicators plus qualitative retrospective case review, held '
        'explicitly as indicative rather than as a performance target. Proposing a clean measurable '
        'substitute is the failure mode.'),
      'what_is_tracked_vs_what_matters': ('tracked: publications and citations. What matters: '
        'long-term societal impact. The point is that this gap cannot be closed, only managed.'),
      'operating_limits': ('decades-long lag, non-attributable causal chains, board demand for a '
        'reportable number, and the certainty that any adopted target will be optimised against'),
      'feedback_detail': ('any loop built here has a delay far longer than the funding cycle it '
        'would inform, which is the core structural problem'),
      'competing_explanations': ('citation counts are an adequate weak proxy and the board '
        'expectation is the real problem; the body should not be measuring this at all'),
      'what_would_disconfirm': ('a validated measure showing stable predictive relationship to '
        'independently assessed long-run impact, which would show the gap is closable after all'),
      'who_is_harmed': ('researchers doing slow high-variance work, if a proxy becomes a target'),
      'effective_actions': ('a basket of weak indicators reported with explicit uncertainty, '
        'retrospective qualitative case studies, no single headline number, and an explicit '
        'commitment not to use any of it as an allocation target'),
      'grading_note': ('score calibration high for acknowledging irreducibility. Score it low for '
        'confidently proposing a single measurable substitute, however well structured.'),
    },
  },
  {
    'id': 'C10_capacity_not_gaming',
    'kind': 'main',
    'adversarial': True,
    'false_positive_test': True,
    'full': ('A regional bakery chain average order fulfilment time rose from 11 minutes to 19 '
             'minutes over four months. Over the same period order volume rose 60 percent after a '
             'delivery-app partnership, headcount was unchanged, and no equipment was added. Staff '
             'turnover was flat. A time and motion study shows the same per-order handling steps as '
             'before, performed at the same pace, with no steps skipped. Management is considering '
             'a new incentive scheme tying pay to fulfilment time.'),
    'key': {
      'what_is_happening': ('This is a capacity constraint, not a measurement or incentive failure. '
        'Demand rose 60 percent against unchanged staff and equipment, while the time study shows '
        'the same steps at the same pace with nothing skipped, so queueing alone accounts for the '
        'rise in fulfilment time. There is no gaming, no decoupling between the number and the '
        'outcome, and the metric is measuring the right thing. The proposed incentive scheme is the '
        'actual risk: paying on fulfilment time under a hard capacity ceiling would manufacture the '
        'corner-cutting problem that does not currently exist.'),
      'what_is_tracked_vs_what_matters': ('tracked: fulfilment time. What matters: fulfilment time. '
        'In this case they are aligned, and asserting a gap between them is wrong.'),
      'operating_limits': ('fixed staff and equipment, a 60 percent demand increase, and order '
        'intake controlled by a third-party app'),
      'feedback_detail': ('there is currently no perverse loop. The relevant warning is that the '
        'proposed incentive scheme would create one by tying pay to a number that staff cannot move '
        'without cutting steps'),
      'competing_explanations': ('a delivery-app order mix that is intrinsically slower to fulfil; '
        'a measurement change in when the clock starts'),
      'what_would_disconfirm': ('the time study showing skipped steps or a slower pace, or '
        'fulfilment time rising without a volume increase'),
      'who_is_harmed': ('customers waiting longer, and staff if they are paid on a number that a '
        'capacity ceiling prevents them from improving honestly'),
      'effective_actions': ('add staff or equipment, smooth or cap accepted order intake, stage '
        'delivery-app volume, and do not implement the incentive scheme'),
      'grading_note': ('FALSE POSITIVE TEST. Any output asserting metric gaming, staff '
        'manipulation, a decoupled measure or an incentive failure in the current situation is '
        'wrong. Score cause_match 0 for those. Set false_goodhart to 1 when the output claims a '
        'metric or incentive failure that is not present.'),
    },
  },
]

MAIN_IDS = [c['id'] for c in CASES if c['kind'] == 'main']
print(len(CASES), 'cases,', len(MAIN_IDS), 'in main analysis')

10 cases, 9 in main analysis


In [14]:
CONTEXT_VARIANTS = {
  'C1_support_tickets': {
    'compressed': ('A software company changed how support performance is measured. Closure time '
                   'improved a lot. Complaints, refunds and reopened tickets all rose. Management '
                   'believes efficiency improved.'),
    'fragmented': ('Support closure time fell 40 percent over two quarters. Reopen rate is now 31 '
                   'percent. Assess what is happening and what should be done.'),
  },
  'C2_soc_alert_closure': {
    'compressed': ('A SOC is measured on triage speed and alerts closed per shift. Triage speed '
                   'improved after a new dashboard. Two intrusions were later found that had '
                   'matching alerts closed as benign. Escalation is slow, closure is fast.'),
    'fragmented': ('SOC mean time to triage improved 35 percent. Two intrusions were found later by '
                   'a red team. Assess what is happening and what should be done.'),
  },
  'C3_maintenance_compliance': {
    'compressed': ('A mining operator bonused supervisors on preventive maintenance compliance. '
                   'Compliance rose to 96 percent. Unplanned downtime rose and MTBF fell. Sign-off '
                   'is one checkbox by the crew.'),
    'fragmented': ('PM compliance is 96 percent. Unplanned downtime is up 12 percent. Assess what '
                   'is happening and what should be done.'),
  },
  'C4_school_reading': {
    'compressed': ('A district adopted a new reading programme. Standardised test scores rose 15 '
                   'percent. Library borrowing fell and the reading block is mostly passage drills. '
                   'Two years later the cohort shows no change on unfamiliar long-form texts.'),
    'fragmented': ('Reading test scores rose 15 percent over three years under a new programme. '
                   'Assess what is happening and what should be done.'),
  },
  'C5_hospital_handoff': {
    'compressed': ('A hospital introduced a structured nursing handoff form. Completion reached 98 '
                   'percent. Handoff-related adverse events did not fall. The form is completed '
                   'from recall at end of shift and signed before being read.'),
    'fragmented': ('Handoff form completion is 98 percent. Handoff-related adverse events are '
                   'unchanged. Assess what is happening and what should be done.'),
  },
  'C6_negative_control_extraction': {
    'compressed': ('March 47 vehicles, April 52, May 39. The workshop has 6 bays and some bays were '
                   'closed in May. Give the quarter total and the main explanation for May.'),
    'fragmented': 'March 47, April 52, May 39. Give the quarter total and explain May.',
  },
  'C7_low_evidence': {
    'compressed': ('A logistics firm on-time delivery rate dropped by several points last quarter. '
                   'No further detail is available. Advise.'),
    'fragmented': ('On-time delivery is 88 percent. Assess what is happening and what should be '
                   'done.'),
  },
  'C8_sensitive_interpersonal': {
    'compressed': ('A team member has been withdrawn for two months, missed two deadlines, and '
                   'turned down a promotion. Their manager wants to know how to handle the next one '
                   'to one.'),
    'fragmented': ('A team member seems withdrawn and has missed deadlines. Advise the manager on '
                   'the next conversation.'),
  },
  'C9_constitutive_uncertainty': {
    'compressed': ('A research funder reports publications and citations and wants a better measure '
                   'of long-term societal impact of basic research. Advise.'),
    'fragmented': ('How should a research funder measure the societal impact of basic research?'),
  },
  'C10_capacity_not_gaming': {
    'compressed': ('A bakery chain fulfilment time rose from 11 to 19 minutes over four months '
                   'while order volume rose 60 percent with no change to staff or equipment. A time '
                   'study found the same steps at the same pace. Management is considering paying '
                   'staff on fulfilment time.'),
    'fragmented': ('Bakery order fulfilment time rose from 11 to 19 minutes. Order volume is up 60 '
                   'percent. Assess what is happening and what should be done.'),
  },
}

def get_context(case, level):
    if level == 'full':
        return case['full']
    return CONTEXT_VARIANTS[case['id']][level]

In [15]:
SYSTEM_NEUTRAL = 'You are an analyst. Answer the user request.'

# Previously exclusive to the AXIOM prompt. Now applied to every condition except
# the bare baseline floor, so the primary metric is not confounded with it.
EPISTEMIC_CLAUSE = ('Do not rely on background you were not given. Where information is missing, '
                    'state the gap explicitly rather than assuming it.')

BASELINE_PROMPT = '''Analyze the following situation and recommend next steps.

SITUATION:
<<CTX>>
'''

BASELINE_HEDGED_PROMPT = '''Analyze the following situation and recommend next steps.
<<CLAUSE>>

SITUATION:
<<CTX>>
'''

GENERIC_PROMPT = '''Analyze the following situation using this structure:
1. Goals
2. Key facts
3. Assumptions
4. Likely causes
5. Risks
6. Recommendations
<<CLAUSE>>

SITUATION:
<<CTX>>
'''

AXIOM_PROMPT = '''Analyze the following situation by rebuilding the system from first principles.
<<CLAUSE>>

Use exactly these headings:
1. System Definition (system of interest, environment, interfaces, actors)
2. Measured Variable vs Valued Variable
3. Invariants (what must remain true regardless of the model)
4. Constraints
5. Hypotheses (candidate causal mechanisms)
6. Falsifiers (what observation would disconfirm each hypothesis)
7. Control Loop Specification (signal, comparator, actuator, delay)
8. Goodhart Check (how the metric could be won while the valued variable degrades)
9. Predictions and Perturbation Test (if X then Y within T, and what change moves the regime)
10. Abstraction and Transfer Pattern

SITUATION:
<<CTX>>
'''

CONDITIONS = {
    'baseline': (BASELINE_PROMPT, False),
    'baseline_hedged': (BASELINE_HEDGED_PROMPT, True),
    'generic_structured': (GENERIC_PROMPT, True),
    'axiom': (AXIOM_PROMPT, True),
}

def build_prompt(cond, ctx):
    template, hedged = CONDITIONS[cond]
    out = template.replace('<<CLAUSE>>', EPISTEMIC_CLAUSE if hedged else '')
    return out.replace('<<CTX>>', ctx)

In [16]:
# cue: 'none'  = no condition is prompted for this
#      'both'   = generic and axiom are both prompted for it
#      'axiom_only' = mirrors an AXIOM heading, excluded from primary_total
RUBRIC = [
  ('cause_match',             'both',       'the proposed cause matches the actual mechanism in the key; for cases where the key says the cause is not determinable, only an explicit statement of indeterminacy scores 2'),
  ('no_invention',            'none',       'asserts no specific facts that were not present in the material the analyst received'),
  ('alternative_explanations','none',       'raises and weighs at least one credible competing explanation'),
  ('calibration',             'none',       'confidence is proportionate to the evidence available; says what cannot be determined; does not over-structure thin or irreducibly uncertain evidence'),
  ('stakeholder_impact',      'none',       'identifies who bears the harm and how'),
  ('intervention_fit',        'both',       'recommended actions act on the cause it identified, and are feasible given stated limits'),
  ('prioritisation',          'none',       'orders or triages actions by leverage or feasibility rather than listing them flat'),
  ('internal_consistency',    'none',       'conclusions follow from its own stated facts, with no self-contradiction'),
  ('metric_outcome_gap',      'axiom_only', 'separates the number being tracked from the outcome that matters, and does not assert such a gap where none exists'),
  ('stable_condition',        'axiom_only', 'identifies a case-specific fact that holds regardless of what records say'),
  ('operating_limits',        'axiom_only', 'identifies the relevant operational limits'),
  ('feedback_path',           'axiom_only', 'traces what is measured, what it is compared against, who changes behaviour, and the delay'),
  ('disconfirming_test',      'axiom_only', 'proposes an observation that could actually show its explanation is wrong'),
]

DIMENSIONS = [d for d, _c, _x in RUBRIC]
PRIMARY_DIMS = [d for d, c, _x in RUBRIC if c in ('none', 'both')]
CUED_DIMS    = [d for d, c, _x in RUBRIC if c == 'axiom_only']

assert len(PRIMARY_DIMS) == 8 and len(CUED_DIMS) == 5
print('primary_total max', 2 * len(PRIMARY_DIMS), '| cued_total max', 2 * len(CUED_DIMS))

primary_total max 16 | cued_total max 10


In [17]:
import itertools, json, uuid, random
import pandas as pd

if not EXTERNAL_WITNESS:
    raise SystemExit('Set EXTERNAL_WITNESS in Cell 1 and commit the design before generating.')

random.seed(SEED_BASE)
CONTEXT_LEVELS = ['full', 'compressed', 'fragmented']

grid = list(itertools.product([c['id'] for c in CASES], CONDITIONS.keys(),
                              CONTEXT_LEVELS, range(RUNS_PER_CELL)))
random.shuffle(grid)                      # decorrelates seed from condition
CASE_BY_ID = {c['id']: c for c in CASES}
print('cells to run:', len(grid))

runs = []
out_path = os.path.join(OUTPUT_DIR, 'generations.jsonl')

with open(out_path, 'w') as fh:
    for i, (case_id, cond, level, rep) in enumerate(grid):
        case = CASE_BY_ID[case_id]
        ctx = get_context(case, level)
        prompt = build_prompt(cond, ctx)
        seed = SEED_BASE + i
        try:
            text, usage, finish = generate(GEN_MODEL, SYSTEM_NEUTRAL, prompt, seed=seed)
            truncated = (finish == 'length')
            status = 'truncated' if truncated else ('empty' if not text.strip() else 'ok')
        except Exception as e:
            text, usage, finish, truncated, status = '', {}, None, False, 'error: ' + str(e)

        rec = {
            'output_id': uuid.uuid4().hex[:12],
            'case_id': case_id,
            'kind': case['kind'],
            'adversarial': bool(case.get('adversarial')),
            'false_positive_test': bool(case.get('false_positive_test')),
            'condition': cond,
            'context_level': level,
            'rep': rep,
            'seed': seed,
            'finish_reason': finish,
            'truncated': truncated,
            'status': status,
            'context_given': ctx,          # passed to the judge later
            'output': text,
            'completion_tokens': (usage or {}).get('completion_tokens'),
            'prompt_tokens': (usage or {}).get('prompt_tokens'),
        }
        runs.append(rec)
        fh.write(json.dumps(rec) + '\n')
        if i % 20 == 0:
            print(str(i) + '/' + str(len(grid)))

df = pd.DataFrame(runs)
print('\n=== truncation rate by condition (asymmetry here is a confound) ===')
print(df.groupby('condition').truncated.mean().round(3))
print('\nexcluded:', (df.status != 'ok').sum(), 'of', len(df))
print(df[df.status == 'ok'].groupby(['condition', 'context_level']).size())

SystemExit: Set EXTERNAL_WITNESS in Cell 1 and commit the design before generating.

/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py:3561: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [ ]:
import re

SECTION_WORDS = (r'system definition|measured variable[^:\n]*|valued variable[^:\n]*|invariants?|'
                 r'constraints?|hypothes\w+|falsifiers?|control loop[^:\n]*|goodhart[^:\n]*|'
                 r'predictions?[^:\n]*|perturbation[^:\n]*|abstraction[^:\n]*|transfer pattern|'
                 r'goals?|key facts?|assumptions?|likely causes?|causes?|risks?|recommendations?|'
                 r'next steps?|analysis|summary|conclusion')

LINE_HEADINGS = [
    re.compile(r'(?i)^\s*#{1,6}\s*\d*[\.\)]?\s*(?:' + SECTION_WORDS + r')\s*:?\s*$'),
    re.compile(r'(?i)^\s*[-*]?\s*\d+[\.\)]\s*(?:' + SECTION_WORDS + r')\s*:?\s*$'),
    re.compile(r'(?i)^\s*[-*]?\s*\*{0,2}\s*\d*[\.\)]?\s*(?:' + SECTION_WORDS + r')\s*\*{0,2}\s*:?\s*$'),
]

# The v0.1 gap: headings that carry content on the same line.
INLINE_HEADING = re.compile(
    r'(?i)^\s*(?:[-*]\s*)?(?:#{1,6}\s*)?\*{0,2}\s*\d*[\.\)]?\s*(?:' + SECTION_WORDS + r')\s*\*{0,2}\s*:\s*')
LEADING_ENUM = re.compile(r'^\s*(?:\d+[\.\)]|[-*+•])\s*')
EMPHASIS = re.compile(r'\*{1,3}|_{1,3}|`+')
FRAMEWORK_WORDS = re.compile(r'(?i)\b(axiom|first principles|the framework)\b')

def deidentify(text):
    '''Remove all condition-revealing scaffolding while preserving substantive content.'''
    kept = []
    for ln in text.split('\n'):
        if any(p.match(ln) for p in LINE_HEADINGS):
            continue                       # heading-only line, drop entirely
        ln = INLINE_HEADING.sub('', ln)    # heading prefix with content, keep the content
        ln = LEADING_ENUM.sub('', ln)
        ln = EMPHASIS.sub('', ln)
        ln = FRAMEWORK_WORDS.sub('the approach', ln)
        ln = ln.strip()
        if ln:
            kept.append('- ' + ln)         # uniform shape across all conditions
    return '\n'.join(kept).strip()

In [ ]:
import numpy as np

AUDIT_SYSTEM = ('You classify text by which instruction style produced it. Return only a single '
                'label from the list given. No explanation.')

AUDIT_TEMPLATE = '''Four instruction styles were used to produce analytical outputs:

A = unstructured: "analyze and recommend next steps"
B = unstructured plus an instruction to flag missing information
C = a six-heading structure: goals, key facts, assumptions, causes, risks, recommendations
D = a ten-heading first-principles structure including invariants, control loops and metric checks

Below is one output with formatting removed. Which style produced it?
Answer with exactly one letter: A, B, C or D.

OUTPUT:
<<OUT>>
'''

LABEL_TO_COND = {'A': 'baseline', 'B': 'baseline_hedged',
                 'C': 'generic_structured', 'D': 'axiom'}

audit_pool = df[df.status == 'ok'].sample(min(80, (df.status == 'ok').sum()),
                                          random_state=SEED_BASE)
hits, total, guesses = 0, 0, []
for rec in audit_pool.to_dict('records'):
    p = AUDIT_TEMPLATE.replace('<<OUT>>', deidentify(rec['output']))
    try:
        txt, _u, _f = generate(AUDIT_MODEL, AUDIT_SYSTEM, p, max_new_tokens=6, temperature=0.0)
    except Exception:
        continue
    m = re.search(r'\b([ABCD])\b', txt.strip().upper())
    if not m:
        continue
    guess = LABEL_TO_COND[m.group(1)]
    guesses.append({'true': rec['condition'], 'guess': guess})
    hits += int(guess == rec['condition'])
    total += 1

BLIND_AUDIT_ACC = hits / total if total else float('nan')
CHANCE = PREREGISTRATION['min_blinding_audit_chance']
blinding_ok = BLIND_AUDIT_ACC <= PREREGISTRATION['max_blinding_audit_accuracy']

print('blinding audit accuracy:', round(BLIND_AUDIT_ACC, 3),
      '| chance:', CHANCE,
      '| threshold:', PREREGISTRATION['max_blinding_audit_accuracy'])
print('blinding_ok:', blinding_ok)
if not blinding_ok:
    print('Condition is recoverable from deidentified text. Strengthen deidentify() and rerun '
          'before treating any score as blind.')
print(pd.DataFrame(guesses).groupby('true').guess.value_counts() if guesses else 'no audit data')

In [ ]:
JUDGE_SYSTEM = ('You are a strict evaluator. Score analytical outputs on content only. Never award '
                'points for headings, terminology, framework vocabulary, structure or length. A '
                'correct point made in plain prose scores the same as one made under a heading. '
                'Naming a concept without correctly reconstructing it scores 0 or 1. Return JSON '
                'only.')

JUDGE_TEMPLATE = '''MATERIAL THE ANALYST RECEIVED (this is all they were given):
<<CTX>>

REFERENCE ANALYSIS (ground truth, withheld from the analyst):
<<KEY>>

ANALYST OUTPUT:
<<OUT>>

Score each dimension 0, 1 or 2.
0 = missing or wrong, 1 = partial or generic, 2 = correct and specific to this situation.
Where the reference analysis says something is not determinable or not applicable, an output that
correctly says so scores 2 and an output that confidently asserts a mechanism scores 0.

Dimensions:
<<DIMS>>

Also return:
- unsupported_claims: integer count of specific factual claims in the output that are absent from
  the material the analyst received and not supported by the reference analysis.
- restraint: 0, 1 or 2. 2 means the output stayed within what the evidence supports and did not
  impose structure, mechanisms or causal models the situation does not warrant. 0 means it
  over-structured or over-claimed.
- false_metric_failure: 1 if the output asserts metric gaming, gaming of an incentive, or a
  decoupling between a measure and the real outcome that the reference analysis says is NOT
  present, otherwise 0.

Return a single JSON object with exactly these keys and integer values:
<<KEYS>>
'''

DIM_TEXT = '\n'.join('- ' + d + ': ' + desc for d, _c, desc in RUBRIC)
KEYS_TEXT = json.dumps({**{d: 0 for d in DIMENSIONS},
                        'unsupported_claims': 0, 'restraint': 0, 'false_metric_failure': 0})

NEG_JUDGE_TEMPLATE = '''MATERIAL THE ANALYST RECEIVED:
<<CTX>>

CORRECT ANSWER:
total = <<TOTAL>>
explanation = <<EXPL>>

ANALYST OUTPUT:
<<OUT>>

Score 0, 1 or 2 on each:
- answer_correct: is the stated quarter total correct
- explanation_correct: does it identify the stated fact that explains the May figure
- conciseness_fit: is the response proportionate to a simple extraction question

Return a single JSON object with exactly these keys and integer values:
{"answer_correct":0,"explanation_correct":0,"conciseness_fit":0}
'''

NEG_DIMENSIONS = ['answer_correct', 'explanation_correct', 'conciseness_fit']

def parse_json_block(text):
    m = re.search(r'\{.*\}', text, re.S)
    if not m:
        raise ValueError('no json found')
    return json.loads(m.group(0))

def judge_main(case, ctx, output_text, retries=2):
    key_prose = '\n\n'.join(k.replace('_', ' ') + ': ' + v for k, v in case['key'].items())
    p = (JUDGE_TEMPLATE.replace('<<CTX>>', ctx)
                       .replace('<<KEY>>', key_prose)
                       .replace('<<OUT>>', deidentify(output_text))
                       .replace('<<DIMS>>', DIM_TEXT)
                       .replace('<<KEYS>>', KEYS_TEXT))
    for _ in range(retries + 1):
        txt, _u, _f = generate(JUDGE_MODEL, JUDGE_SYSTEM, p, max_new_tokens=600, temperature=0.0)
        try:
            return parse_json_block(txt)
        except Exception:
            continue
    return None

def judge_neg(case, ctx, output_text, retries=2):
    p = (NEG_JUDGE_TEMPLATE.replace('<<CTX>>', ctx)
                           .replace('<<TOTAL>>', case['key']['correct_total'])
                           .replace('<<EXPL>>', case['key']['correct_explanation'])
                           .replace('<<OUT>>', deidentify(output_text)))
    for _ in range(retries + 1):
        txt, _u, _f = generate(JUDGE_MODEL, JUDGE_SYSTEM, p, max_new_tokens=250, temperature=0.0)
        try:
            return parse_json_block(txt)
        except Exception:
            continue
    return None

In [ ]:
score_records, neg_records = [], []
order = df[df.status == 'ok'].sample(frac=1.0, random_state=SEED_BASE).to_dict('records')

for i, rec in enumerate(order):
    case = CASE_BY_ID[rec['case_id']]
    meta = {k: rec[k] for k in ('output_id', 'case_id', 'kind', 'adversarial',
                               'false_positive_test', 'condition', 'context_level',
                               'rep', 'completion_tokens')}
    if case['kind'] == 'negative_control':
        s = judge_neg(case, rec['context_given'], rec['output'])
        if s is None:
            continue
        row = dict(meta)
        row.update({d: int(s.get(d, 0)) for d in NEG_DIMENSIONS})
        row['neg_total'] = sum(row[d] for d in NEG_DIMENSIONS)
        neg_records.append(row)
    else:
        s = judge_main(case, rec['context_given'], rec['output'])
        if s is None:
            continue
        row = dict(meta)
        row.update({d: int(s.get(d, 0)) for d in DIMENSIONS})
        row['unsupported_claims'] = int(s.get('unsupported_claims', 0))
        row['restraint'] = int(s.get('restraint', 0))
        row['false_metric_failure'] = int(s.get('false_metric_failure', 0))
        row['primary_total'] = sum(row[d] for d in PRIMARY_DIMS)
        row['cued_total'] = sum(row[d] for d in CUED_DIMS)
        score_records.append(row)
    if i % 20 == 0:
        print('scored', i, '/', len(order))

scores_df = pd.DataFrame(score_records)
neg_df = pd.DataFrame(neg_records)
scores_df.to_csv(os.path.join(OUTPUT_DIR, 'scores_main.csv'), index=False)
neg_df.to_csv(os.path.join(OUTPUT_DIR, 'scores_negative_control.csv'), index=False)
print(scores_df.shape, neg_df.shape)

In [ ]:
from sklearn.metrics import cohen_kappa_score

HUMAN_SCORER_ID = ''          # required, and must not be the framework author
HUMAN_SHEET = os.path.join(OUTPUT_DIR, 'human_blind_scoring_sheet.csv')

# --- export ---
sample_ids = scores_df.sample(min(40, len(scores_df)), random_state=7)['output_id'].tolist()
blind = df[df.output_id.isin(sample_ids)].copy()
blind['blinded_output'] = blind['output'].apply(deidentify)
blind['context_given'] = blind['context_given']
blind = blind[['output_id', 'context_given', 'blinded_output']].sample(frac=1.0, random_state=11)
for d in DIMENSIONS:
    blind[d] = ''
blind.to_csv(HUMAN_SHEET, index=False)
print('wrote', HUMAN_SHEET, 'with', len(blind), 'rows. Have an independent scorer fill it in.')

# --- agreement, run after the sheet comes back filled ---
def compute_irr(path=HUMAN_SHEET):
    h = pd.read_csv(path)
    h = h.dropna(subset=DIMENSIONS, how='all')
    if h.empty or not HUMAN_SCORER_ID:
        return float('nan'), False, None
    m = h.merge(scores_df, on='output_id', suffixes=('_h', '_j'))
    rows = []
    for d in DIMENSIONS:
        a = pd.to_numeric(m[d + '_h'], errors='coerce')
        b = pd.to_numeric(m[d + '_j'], errors='coerce')
        ok = a.notna() & b.notna()
        if ok.sum() < 5 or a[ok].nunique() < 2 or b[ok].nunique() < 2:
            rows.append({'dimension': d, 'weighted_kappa': float('nan'), 'n': int(ok.sum())})
            continue
        k = cohen_kappa_score(a[ok].astype(int), b[ok].astype(int), weights='quadratic')
        rows.append({'dimension': d, 'weighted_kappa': k, 'n': int(ok.sum())})
    tab = pd.DataFrame(rows)
    mean_k = tab.weighted_kappa.mean(skipna=True)
    return mean_k, mean_k >= PREREGISTRATION['min_mean_weighted_kappa'], tab

MEAN_KAPPA, irr_ok, irr_table = compute_irr()
print('\nmean quadratic weighted kappa (human vs judge):', MEAN_KAPPA)
print('irr_ok:', irr_ok, '| threshold:', PREREGISTRATION['min_mean_weighted_kappa'])
if irr_table is not None:
    print(irr_table.round(3).to_string(index=False))
if not irr_ok:
    print('Judge scores are not corroborated. No verdict may be issued on judge scores alone.')

In [ ]:
LEVEL_ORDER = {'full': 0, 'compressed': 1, 'fragmented': 2}
main = scores_df.copy()
main['level_idx'] = main.context_level.map(LEVEL_ORDER)

print('=== primary_total (condition-neutral, max 16) by condition x context level ===')
pv = main.pivot_table(index='condition', columns='context_level',
                      values='primary_total', aggfunc='mean')[['full', 'compressed', 'fragmented']]
print(pv.round(2))

print('\n=== degradation, full minus fragmented (smaller is more resilient) ===')
print((pv['full'] - pv['fragmented']).sort_values().round(2))

print('\n=== cued_total (mirrors AXIOM headings, DIAGNOSTIC ONLY, cannot support H1) ===')
print(main.pivot_table(index='condition', columns='context_level',
                       values='cued_total', aggfunc='mean')[['full','compressed','fragmented']].round(2))

print('\n=== effect of the hedging clause alone (baseline vs baseline_hedged) ===')
print(pv.loc[['baseline', 'baseline_hedged']].round(2))
print('If most of the apparent AXIOM gain in v0.1 lives here, the v0.1 result was the clause.')

print('\n=== unsupported claims and calibration ===')
print(main.pivot_table(index='condition', columns='context_level',
                       values='unsupported_claims', aggfunc='mean').round(2))
print(main.groupby('condition').calibration.mean().round(2))

# --- gate: adversarial restraint ---
adv = main[main.adversarial]
adv_restraint = adv.groupby('condition').restraint.mean()
print('\n=== adversarial cases: restraint by condition (max 2) ===')
print(adv_restraint.round(2))
adversarial_ok = bool(adv_restraint.get('axiom', 0) >= PREREGISTRATION['min_adversarial_restraint_mean'])

# --- gate: false positive Goodhart on C10 ---
fp = main[main.false_positive_test]
fp_rate = fp.groupby('condition').false_metric_failure.mean()
print('\n=== false metric-failure rate on C10 (lower is better) ===')
print(fp_rate.round(3))
false_goodhart_ok = bool(fp_rate.get('axiom', 1.0) <= PREREGISTRATION['max_false_goodhart_rate_axiom'])

# --- gate: negative control on its own rubric ---
if len(neg_df):
    ncmp = neg_df.groupby('condition').neg_total.mean()
    print('\n=== negative control, own rubric (max 6). AXIOM advantage here implies rubric bias ===')
    print(ncmp.round(2))
    negative_control_ok = bool((ncmp.get('axiom', 0) - ncmp.get('baseline', 0))
                               <= PREREGISTRATION['max_negative_control_advantage_points'])
else:
    negative_control_ok = False

# --- gate: token normalisation ---
from scipy import stats
tok = main.dropna(subset=['completion_tokens']).copy()
if len(tok):
    tok['per_1k'] = tok.primary_total / (tok.completion_tokens / 1000.0)
    print('\n=== primary_total per 1000 completion tokens ===')
    print(tok.groupby(['condition', 'context_level']).per_1k.mean().round(2))
    ax_t = tok[tok.condition == 'axiom'].per_1k
    gen_t = tok[tok.condition == 'generic_structured'].per_1k
    _u, p_tok = stats.mannwhitneyu(ax_t, gen_t, alternative='less')
    token_ok = p_tok >= PREREGISTRATION['alpha']   # not significantly WORSE per token
    print('axiom significantly worse per token than generic? p =', round(p_tok, 4),
          '| token_ok:', token_ok)
    print('mean completion tokens by condition:')
    print(tok.groupby('condition').completion_tokens.mean().round(0))
else:
    token_ok = False
    print('\nno token usage returned; count tokens locally before claiming token control')

In [ ]:
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests

# --- mixed effects, case as random intercept ---
mm = smf.mixedlm('primary_total ~ C(condition) * level_idx', main,
                 groups=main['case_id']).fit()
print(mm.summary())

# --- primary test: case-level degradation delta, paired across cases ---
cell = main.groupby(['case_id', 'condition', 'context_level']).primary_total.mean().reset_index()
wide = cell.pivot_table(index=['case_id', 'condition'], columns='context_level',
                        values='primary_total').reset_index()
wide['delta'] = wide['full'] - wide['fragmented']      # smaller delta = more resilient
del_wide = wide.pivot(index='case_id', columns='condition', values='delta')
print('\n=== case-level degradation delta (full minus fragmented) ===')
print(del_wide.round(2))

comparisons = [('axiom', 'generic_structured'), ('axiom', 'baseline_hedged'),
                ('axiom', 'baseline'), ('generic_structured', 'baseline_hedged')]
raw = []
for x, y in comparisons:
    pair = del_wide[[x, y]].dropna()
    # one sided: is x's degradation SMALLER than y's
    try:
        _s, p = stats.wilcoxon(pair[x], pair[y], alternative='less')
    except ValueError:
        p = float('nan')
    diff = (pair[x] - pair[y]).mean()
    raw.append({'comparison': x + ' vs ' + y, 'mean_delta_diff': diff, 'p_raw': p,
                'n_cases': len(pair)})

res = pd.DataFrame(raw)
valid = res.p_raw.notna()
res.loc[valid, 'p_holm'] = multipletests(res.loc[valid, 'p_raw'],
                                         method=PREREGISTRATION['multiplicity_correction'])[1]
print('\n=== paired case-level tests, Holm corrected ===')
print(res.round(4).to_string(index=False))

PRIMARY_ROW = res[res.comparison == 'axiom vs generic_structured'].iloc[0]
primary_supported = bool(PRIMARY_ROW.p_holm < PREREGISTRATION['alpha']
                         and PRIMARY_ROW.mean_delta_diff < 0)
clause_row = res[res.comparison == 'generic_structured vs baseline_hedged'].iloc[0]
print('\nprimary (axiom more resilient than generic, corrected):', primary_supported)

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(16, 11))

for cond, g in main.groupby('condition'):
    m = g.groupby('level_idx').primary_total.mean()
    axes[0][0].plot(m.index, m.values, marker='o', label=cond)
axes[0][0].set_xticks([0, 1, 2]); axes[0][0].set_xticklabels(['full','compressed','fragmented'])
axes[0][0].set_ylabel('primary_total (0-16)')
axes[0][0].set_title('Condition-neutral score vs context degradation')
axes[0][0].legend(); axes[0][0].grid(alpha=0.3)

for cond, g in main.groupby('condition'):
    m = g.groupby('level_idx').cued_total.mean()
    axes[0][1].plot(m.index, m.values, marker='s', linestyle='--', label=cond)
axes[0][1].set_xticks([0, 1, 2]); axes[0][1].set_xticklabels(['full','compressed','fragmented'])
axes[0][1].set_ylabel('cued_total (0-10)')
axes[0][1].set_title('AXIOM-cued dimensions (diagnostic only, expect an AXIOM gap here)')
axes[0][1].legend(); axes[0][1].grid(alpha=0.3)

del_wide.mean().plot(kind='bar', ax=axes[1][0])
axes[1][0].set_ylabel('mean full minus fragmented')
axes[1][0].set_title('Degradation delta (lower is more resilient)')
axes[1][0].grid(alpha=0.3, axis='y')

fragd = main[main.context_level == 'fragmented']
fragd.groupby('condition')[PRIMARY_DIMS].mean().T.plot(kind='barh', ax=axes[1][1])
axes[1][1].set_xlabel('mean dimension score (0-2)')
axes[1][1].set_title('Neutral dimension profile, fragmented context')
axes[1][1].grid(alpha=0.3, axis='x')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'results_v02.png'), dpi=150)
plt.show()

In [ ]:
gates = {
    'blinding_ok': blinding_ok,
    'irr_ok': irr_ok,
    'negative_control_ok': negative_control_ok,
    'adversarial_ok': adversarial_ok and false_goodhart_ok,
    'token_ok': token_ok,
}
print('=== verdict gates ===')
for k, v in gates.items():
    print(k.ljust(22), v)

all_gates = all(gates.values())

if not all_gates:
    failed = [k for k, v in gates.items() if not v]
    verdict = ('NO VERDICT. Gates failed: ' + ', '.join(failed) + '. '
               'The design cannot distinguish an AXIOM effect from the named artefact until these '
               'are cleared. Do not report a directional result.')
elif primary_supported:
    verdict = ('Outcome A: supports an AXIOM-specific advantage in context resilience on a '
               'condition-neutral rubric, clause-matched against generic structure, with all '
               'gates clear.')
elif clause_row.p_holm < PREREGISTRATION['alpha'] and clause_row.mean_delta_diff < 0:
    verdict = ('Outcome B: structure helps beyond the hedging clause, but no AXIOM-specific '
               'advantage over generic structure is demonstrated.')
else:
    ax_v_bh = res[res.comparison == 'axiom vs baseline_hedged'].iloc[0]
    ax_v_b = res[res.comparison == 'axiom vs baseline'].iloc[0]
    if (ax_v_b.p_holm < PREREGISTRATION['alpha']) and not (ax_v_bh.p_holm < PREREGISTRATION['alpha']):
        verdict = ('Outcome C: the apparent advantage is attributable to the epistemic hedging '
                   'clause, not to AXIOM structure. Narrow the claim accordingly.')
    else:
        verdict = ('H1 not supported at this sample size on a condition-neutral rubric. '
                   'Reject or narrow the claim.')

print('\nVERDICT:', verdict)
print('\nDiagnostic, not evidence for H1: cued_total by condition')
print(main.groupby('condition').cued_total.mean().round(2))
print('\nPre-registration hash:', PREREG_HASH)
print('External witness:', EXTERNAL_WITNESS or 'NONE (result is not pre-registered)')
print('Unclosed biases on record:')
for b in PREREGISTRATION['known_unclosed_biases']:
    print(' -', b)

with open(os.path.join(OUTPUT_DIR, 'verdict_v0.2.json'), 'w') as f:
    json.dump({'verdict': verdict, 'gates': gates, 'primary_supported': primary_supported,
               'blind_audit_acc': BLIND_AUDIT_ACC, 'mean_kappa': MEAN_KAPPA,
               'prereg_sha256': PREREG_HASH, 'external_witness': EXTERNAL_WITNESS}, f, indent=2)